In [2]:
import requests
import zipfile
from io import BytesIO
import xml.etree.ElementTree as ET
import pandas as pd

url = "https://opendart.fss.or.kr/api/corpCode.xml"
api_key_dart = "52274dad778cf14b88c8a0e18995d7d6929d437b"
params = {'crtfc_key' : api_key_dart}

r = requests.get(url=url, params=params, timeout=10)
data = r.content
with zipfile.ZipFile(BytesIO(data)) as z:
    for name in z.namelist():
        if name.endswith(".xml"):
            with z.open(name) as f:
                root = ET.parse(f).getroot()
                print(name, root.tag)

# 첫 번째 레코드를 기준으로 컬럼명 추출
columns = [elem.tag for elem in root[0]]

# 데이터 저장
data = []
for item in root:
    row = {col: item.find(col).text if item.find(col) is not None else '' for col in columns}
    data.append(row)

# DataFrame 변환
corpcode = pd.DataFrame(data)
corpcode

CORPCODE.xml result


,corp_code,corp_name,corp_eng_name,stock_code,modify_date
0,00434003,다코,Daco corporation,,20170630
1,00430964,굿앤엘에스,"Good & LS Co.,Ltd.",,20170630
2,00388953,크레디피아제이십오차유동화전문회사,Credipia 25th Asset Securitization Specialty L...,,20170630
3,00179984,연방건설산업,youn bao,,20170630
4,00420143,브룩스피알아이오토메이션잉크,"BROOKS-PRI Automation, Inc.",,20170630
...,...,...,...,...,...
115207,00652043,한솔오리온텍,"Hansol Oriontech Co.,Ltd.",,20250918
115208,01956955,도경회계법인,Dokyung Accounting Corporation,,20250918
115209,01956964,새중앙새마을금고,Saejungangsaemaeulgeumgo,,20250918
115210,01956973,현대인베스트선진인더스트리얼일반사모부동산투자회사,Hyundai Investment Sunjin Industrial Private R...,,20250918


In [62]:
corpcode[~(corpcode.stock_code==" ")]

,corp_code,corp_name,corp_eng_name,stock_code,modify_date
1944,00260985,한빛네트,"HanbitNet, Inc.",036720,20170630
1956,00264529,엔플렉스,"Nplex,Inc.",040130,20170630
1957,00358545,동서정보기술,"Dong Seo Information Technology Co., Ltd.",055000,20170630
2695,00231567,애드모바일,,032600,20170630
3765,00359614,리더컴,"LEADER COMM.Co.,Ltd",056140,20170630
...,...,...,...,...,...
115165,01182408,소프트캠프,"SOFTCAMP CO., LTD",258790,20251031
115167,00600013,맵스리얼티,MIRAEASSET MAPS REALTY INVESTMENT COMPANY,094800,20251031
115180,00220057,유비케어,"UBCARE CO., LTD.",032620,20251103
115181,00104722,국제약품,"Kukje Pharma Co., Ltd.",002720,20251103


In [87]:
from pykrx import stock
date = "20250211"  # YYYYMMDD
df_kospi = stock.get_market_fundamental_by_ticker(date, market="KOSPI")
stock_list = df_kospi.index.to_list()
corp_code_list = corpcode[corpcode["stock_code"].isin(stock_list)].corp_code.to_list()
out_stock_list = corpcode[corpcode["stock_code"].isin(stock_list)].stock_code.to_list()
print(corp_code_list, f'input_n : {len(stock_list)}', f'output_n : {len(corp_code_list)}', f'execpt: {[s for s in stock_list if s not in out_stock_list]}', sep='\n')

['00173944', '00109286', '00129350', '00144252', '00303217', '01036446', '00112970', '00129554', '00163691', '00260392', '00105156', '00146649', '00148443', '00148939', '01234297', '00166272', '00143794', '00145260', '00118965', '00118008', '00132354', '00159254', '00168401', '00648721', '01532603', '00111458', '00218575', '00134316', '00103042', '01267684', '00104698', '00122348', '00127200', '00147222', '00145464', '00109161', '00101628', '00823429', '00174004', '00123107', '00220561', '00165343', '00159218', '00101220', '00152686', '00162993', '00124027', '00145686', '00332927', '00398792', '00148920', '00136864', '00137809', '00136457', '00118345', '00121288', '00146269', '00130091', '00138729', '00172936', '00151395', '00153126', '00653024', '00203582', '00165583', '00860332', '00159023', '00138297', '00159795', '00105952', '00160621', '00539274', '01244601', '00108977', '00148522', '00134963', '01133217', '00148948', '00124151', '00157070', '01032486', '00117577', '00153755', '00

In [147]:
dfs = []
for i in range(0, len(corp_code_list), 100):
    chunk_corp_code_list = corp_code_list[i:i+100]

    url = "https://opendart.fss.or.kr/api/fnlttMultiAcnt.json"
    params = {"crtfc_key" : api_key_dart, 'corp_code':','.join(chunk_corp_code_list), 'bsns_year': '2024', 'reprt_code':'11011'} # 1분기보고서 : 11013 / 반기보고서 : 11012 / 3분기보고서 : 11014 / 사업보고서 : 11011
    r = requests.get(url=url, params=params, timeout=10)
    data = r.json()
    print(data['message'])
    df = pd.DataFrame(data['list'])
    dfs.append(df)
fin_data = pd.concat(dfs, ignore_index=True)
cols = ['corp_name', 'corp_code', 'stock_code', 'rcept_no','fs_nm', 'account_nm', 'thstrm_amount', 'currency']
f_fin_data = fin_data.merge(corpcode)[cols]


정상
정상
정상
정상
정상
정상
정상
정상
정상


In [ ]:
f_list = f_fin_data.corp_name.unique().tolist()
len(f_list)

818

In [154]:
f_fin_data['thstrm_amount'] = f_fin_data['thstrm_amount'].replace(',',"", regex=True).apply(pd.to_numeric, errors="coerce")

In [155]:
pd.options.display.float_format = '{:,.0f}'.format
f_fin_data[(f_fin_data['account_nm']=='매출액')&(f_fin_data['fs_nm']=='연결재무제표')].sort_values(by='thstrm_amount', ascending=False).assign(rank=lambda x: x["thstrm_amount"].rank(ascending=False, method="min")).style.format({"thstrm_amount": "{:,.0f}"})

,corp_name,corp_code,stock_code,rcept_no,fs_nm,account_nm,thstrm_amount,currency,rank
19503,삼성전자,00126380,005930,20250311001085,연결재무제표,매출액,"300,870,903,000,000",KRW,1.000000
17652,현대자동차,00164742,005380,20250312001148,연결재무제표,매출액,"175,231,153,000,000",KRW,2.000000
9543,SK,00181712,034730,20250318001364,연결재무제표,매출액,"124,690,439,000,000",KRW,3.000000
2695,기아,00106641,000270,20250313001390,연결재무제표,매출액,"107,448,752,000,000",KRW,4.000000
9217,한국전력공사,00159193,015760,20250318000747,연결재무제표,매출액,"93,398,896,000,000",KRW,5.000000
7082,LG전자,00401731,066570,20250317001029,연결재무제표,매출액,"87,728,182,000,000",KRW,6.000000
18272,SK이노베이션,00631518,096770,20250318000862,연결재무제표,매출액,"74,716,970,174,000",KRW,7.000000
11918,POSCO홀딩스,00155319,005490,20250312001016,연결재무제표,매출액,"72,688,143,160,111",KRW,8.000000
7841,HD현대,01205709,267250,20250319001012,연결재무제표,매출액,"67,765,626,088,000",KRW,9.000000
9402,SK하이닉스,00164779,000660,20250319000665,연결재무제표,매출액,"66,192,960,000,000",KRW,10.000000


In [184]:
dfs = []
for i in range(0, len(corp_code_list), 100):
    chunk_corp_code_list = corp_code_list[i:i+100]

    url = "https://opendart.fss.or.kr/api/fnlttCmpnyIndx.json"
    params = {"crtfc_key" : api_key_dart, 'corp_code':','.join(chunk_corp_code_list), 'bsns_year': '2024', 'reprt_code':'11011', 'idx_cl_code': 'M210000' } # 수익성지표 : M210000 안정성지표 : M220000 성장성지표 : M230000 활동성지표 : M240000
    r = requests.get(url=url, params=params, timeout=10)
    data = r.json()
    print(data['message'])
    df = pd.DataFrame(data['list'])
    dfs.append(df)
index_data = pd.concat(dfs, ignore_index=True)
cols = ['corp_name', 'corp_code', 'stock_code', 'reprt_code','bsns_year', 'stlm_dt', 'idx_cl_nm', 'idx_nm', 'idx_val']
f_index_data = index_data.merge(corpcode)[cols]
f_index_data['idx_val'] = f_index_data['idx_val'].str.replace(",", "").pipe(pd.to_numeric, errors="coerce")
f_index_data

정상
정상
정상
정상
정상
정상
정상
정상
정상


,corp_name,corp_code,stock_code,reprt_code,bsns_year,stlm_dt,idx_cl_nm,idx_nm,idx_val
0,KG케미칼,00101220,001390,11011,2024,2024-12-31,수익성지표,세전계속사업이익률,3
1,KG케미칼,00101220,001390,11011,2024,2024-12-31,수익성지표,순이익률,3
2,KG케미칼,00101220,001390,11011,2024,2024-12-31,수익성지표,총포괄이익률,2
3,KG케미칼,00101220,001390,11011,2024,2024-12-31,수익성지표,매출총이익률,12
4,KG케미칼,00101220,001390,11011,2024,2024-12-31,수익성지표,매출원가율,88
...,...,...,...,...,...,...,...,...,...
12175,DL이앤씨,01524093,375500,11011,2024,2024-12-31,수익성지표,자기자본세전계속사업이익률,NaN
12176,DL이앤씨,01524093,375500,11011,2024,2024-12-31,수익성지표,자본금영업이익률,122
12177,DL이앤씨,01524093,375500,11011,2024,2024-12-31,수익성지표,자본금세전계속사업이익률,NaN
12178,DL이앤씨,01524093,375500,11011,2024,2024-12-31,수익성지표,납입자본이익률,103


In [185]:
l = ['005930', '000660', '373220', '207940', '005380', '005935', '000270', '068270', '005490', '035420']
con = (f_index_data.stock_code.isin(l))
f_index_data[(f_index_data.idx_nm =="매출원가율")].sort_values(by='idx_val', ascending=False).style.format({"idx_val": "{:,.2f}"})

,corp_name,corp_code,stock_code,reprt_code,bsns_year,stlm_dt,idx_cl_nm,idx_nm,idx_val
11348,카프로,00159810,006380,11011,2024,2024-12-31,수익성지표,매출원가율,510.61
11854,SK아이이테크놀로지,01386916,361610,11011,2024,2024-12-31,수익성지표,매출원가율,174.30
10548,티웨이홀딩스,00109514,004870,11011,2024,2024-12-31,수익성지표,매출원가율,139.92
11989,진원생명과학,00118521,011000,11011,2024,2024-12-31,수익성지표,매출원가율,130.43
3312,범양건영,00122694,002410,11011,2024,2024-12-31,수익성지표,매출원가율,125.89
11318,엑시큐어하이트론,00156150,019490,11011,2024,2024-12-31,수익성지표,매출원가율,125.27
11554,엘앤에프,00398701,066970,11011,2024,2024-12-31,수익성지표,매출원가율,124.30
9680,참엔지니어링,00153621,009310,11011,2024,2024-12-31,수익성지표,매출원가율,118.76
7156,화인베스틸,00661847,133820,11011,2024,2024-12-31,수익성지표,매출원가율,111.28
10428,에이프로젠바이오로직스,00101044,003060,11011,2024,2024-12-31,수익성지표,매출원가율,111.25


In [186]:
index_data

,reprt_code,bsns_year,corp_code,stock_code,stlm_dt,idx_cl_code,idx_cl_nm,idx_code,idx_nm,idx_val
0,11011,2024,00101220,001390,2024-12-31,M210000,수익성지표,M211100,세전계속사업이익률,2.578
1,11011,2024,00101220,001390,2024-12-31,M210000,수익성지표,M211200,순이익률,2.578
2,11011,2024,00101220,001390,2024-12-31,M210000,수익성지표,M211250,총포괄이익률,2.046
3,11011,2024,00101220,001390,2024-12-31,M210000,수익성지표,M211300,매출총이익률,11.555
4,11011,2024,00101220,001390,2024-12-31,M210000,수익성지표,M211400,매출원가율,88.445
...,...,...,...,...,...,...,...,...,...,...
12175,11011,2024,01524093,375500,2024-12-31,M210000,수익성지표,M212300,자기자본세전계속사업이익률,NaN
12176,11011,2024,01524093,375500,2024-12-31,M210000,수익성지표,M212400,자본금영업이익률,122.077
12177,11011,2024,01524093,375500,2024-12-31,M210000,수익성지표,M212500,자본금세전계속사업이익률,NaN
12178,11011,2024,01524093,375500,2024-12-31,M210000,수익성지표,M212600,납입자본이익률,103.28


In [227]:
url = "https://opendart.fss.or.kr/api/list.json"
api_key_dart = "52274dad778cf14b88c8a0e18995d7d6929d437b"
params = {'crtfc_key' : api_key_dart, 'corp_code': '00126487', 'bgn_de':'20230101', 'end_de': '20250101', 'pblntf_ty': 'A'} # A : 정기공시/ B : 주요사항보고 / C : 발행공시 / D : 지분공시 / E : 기타공시 / F : 외부감사관련 / G : 펀드공시 / H : 자산유동화 / I : 거래소공시 / J : 공정위공시
r = requests.get(url=url, params=params, timeout=10)
report_df = pd.DataFrame(r.json()['list'])
report_df

,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,00126487,F&F 홀딩스,007700,Y,분기보고서 (2024.09),20241114002459,F&F 홀딩스,20241114,
1,00126487,F&F 홀딩스,007700,Y,반기보고서 (2024.06),20240814004062,F&F 홀딩스,20240814,
2,00126487,F&F 홀딩스,007700,Y,분기보고서 (2024.03),20240516001408,F&F 홀딩스,20240516,
3,00126487,F&F 홀딩스,007700,Y,사업보고서 (2023.12),20240320001670,F&F 홀딩스,20240320,연
4,00126487,F&F 홀딩스,007700,Y,분기보고서 (2023.09),20231114002046,F&F 홀딩스,20231114,
5,00126487,F&F 홀딩스,007700,Y,반기보고서 (2023.06),20230814002273,F&F 홀딩스,20230814,
6,00126487,F&F 홀딩스,007700,Y,분기보고서 (2023.03),20230515001870,F&F 홀딩스,20230515,
7,00126487,F&F 홀딩스,007700,Y,사업보고서 (2022.12),20230321001401,F&F 홀딩스,20230321,연


In [222]:
import requests
import zipfile
from io import BytesIO
import xml.etree.ElementTree as ET
import pandas as pd

url = "https://opendart.fss.or.kr/api/fnlttXbrl.xml"
api_key_dart = "52274dad778cf14b88c8a0e18995d7d6929d437b"
params = {'crtfc_key' : api_key_dart, 'rcept_no': '20240320001885', 'reprt_code':'11011'} # 	1분기보고서 : 11013 / 반기보고서 : 11012 / 3분기보고서 : 11014 / 사업보고서 : 11011

r = requests.get(url=url, params=params, timeout=10)
data = r.content
with open("dart_xbrl.zip", "wb") as f:
    f.write(data)

print("저장 완료")

저장 완료


In [228]:
from __future__ import annotations

import re
import zipfile
from typing import Dict, List, Tuple, Optional

from lxml import etree

def _decode_bytes(b: bytes) -> str:
    """
    DART 문서가 UTF-8이거나 EUC-KR/CP949인 경우가 많아서 순차 시도.
    """
    for enc in ("utf-8", "cp949", "euc-kr"):
        try:
            return b.decode(enc)
        except UnicodeDecodeError:
            continue
    # 최후: 깨진 문자는 대체
    return b.decode("utf-8", errors="replace")


def _normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def split_sentences(text: str) -> str:
    """
    한국어 문장을 기준으로 한 줄에 한 문장씩 정렬
    """
    # 마침표, 물음표, 느낌표 뒤에서 줄바꿈
    sentences = re.split(r'(?<=[\.!?])\s+', text)
    
    # 공백 정리 + 빈 문장 제거
    sentences = [s.strip() for s in sentences if s.strip()]
    
    return "\n".join(sentences)

doc_url = "https://opendart.fss.or.kr/api/document.xml"
doc_params = {"crtfc_key": api_key_dart, "rcept_no": '20240320001670'}

r = requests.get(doc_url, params=doc_params, timeout=60)
r.raise_for_status()

target = None
target_text = None

with zipfile.ZipFile(BytesIO(r.content)) as zf:
    for name in zf.namelist():
        xml_names = [n for n in zf.namelist() if n.lower().endswith(".xml")]
        print(xml_names)
        for name in xml_names:
            raw = zf.read(name)
            text = _decode_bytes(raw)
        if ('ACODE="11011"' in text) and ("사업보고서" in text):
            target = name
            target_text = text
            break
        if target is None:
            # 그래도 못찾으면 조금 완화해서 ACODE만 찾기
            for name in xml_names:
                raw = zf.read(name)
                text = _decode_bytes(raw)
                if 'ACODE="11011"' in text:
                    target = name
                    target_text = text
                    break

parser = etree.XMLParser(recover=True, huge_tree=True)
root = etree.fromstring(target_text.encode("utf-8", errors="ignore"), parser=parser)

    # 문서 내 TITLE 노드들을 문서 순서대로 수집
titles = root.xpath(".//TITLE")
if not titles:
    raise RuntimeError("XML에서 <TITLE> 노드를 찾지 못했습니다. (문서 구조가 다를 수 있음)")
title_texts = [_normalize_text("".join(t.itertext())) for t in titles]
wanted_titles = ["1. 사업의 개요", "2. 주요 제품 및 서비스"]
wanted_set = set(_normalize_text(t) for t in wanted_titles)
found_indices = [i for i, t in enumerate(title_texts) if t in wanted_set]
found_indices

results: Dict[str, str] = {}

    # “TITLE 다음부터 다음 TITLE 전까지” 텍스트 수집
    # 방법: root를 문서 순서로 순회하며 TITLE 노드를 만나면 상태를 바꾸는 방식이 안정적
    # (TITLE이 중첩/레벨이 다양해도, 일단 '다음 TITLE 전까지'라는 요구에 맞음)
current_title: Optional[str] = None
capture = False
buf: List[str] = []

wanted_titles_norm = set(_normalize_text(t) for t in wanted_titles)

# root 전체를 document-order로 순회
for node in root.iter():
    if node.tag == "TITLE":
        # TITLE 도착 -> 이전 TITLE 캡처 종료
        if capture and current_title is not None:
            results[current_title] = _normalize_text(" ".join(buf))
        # 새 TITLE 시작
        current_title = _normalize_text("".join(node.itertext()))
        capture = current_title in wanted_titles_norm
        buf = []
        continue

    # 캡처 중이면 텍스트를 긁어모으기
    if capture:
        # node.text / node.tail 모두 고려
        if node.text and node.text.strip():
            buf.append(node.text.strip())
        if node.tail and node.tail.strip():
            buf.append(node.tail.strip())

    # 마지막 구간 저장
    if capture and current_title is not None:
        results[current_title] = _normalize_text(" ".join(buf))

    # wanted_titles에 대해 누락이 있을 수 있으니 필터링/정렬
    result = {t: results.get(_normalize_text(t), "") for t in wanted_titles}
for k, v in result.items():
        print("\n" + "=" * 80)
        print(k)
        print("=" * 80)
        formatted = split_sentences(v)
        print(formatted) 

['20240320001670_00760.xml', '20240320001670_00761.xml', '20240320001670.xml']

1. 사업의 개요
□ 지주회사[에프앤에프홀딩스] 지주회사(持株會社, Holding Company)란 다른 회사의 주식을 소유함으로서 그 회사의 사업내용을 지배하는 것을 주된 사업으로 하는 회사를 말합니다.
사업내용을 지배한다는 것은 회사의 사업에 관한 주요 경영사항에 관여하고, 그에 관한 결정에 영향력을 행사한다는 것을 의미합니다.지주회사에 대한 법률적 근거는 공정거래법의 제2조 제1호의 2에서 그 기준을 정하고 있습니다.
지주회사는 회사가 소유하고 있는 자산총액이 5천억원 이상으로서 지배 목적의 자회사에 대한 주식가액(지분포함) 합계액이 당해 회사 자산총액의 100분의 50 이상인 회사를 의미합니다.
지주회사 수는 '99년 제도 도입 이후 꾸준히 증가하다 17년 9월 이후 감소추세였으나, 22년부터 다시 증가하는 추세로 '23년 9월 말 기준 국내 지주회사의 수는 총 172개사 입니다.
※ 연도별 지주회사 현황 (단위 : 개사,%) 구 분 '23.9 '22.9 '21.9 '20.9 '19.9 '18.9 '17.9 '16.9 '15.9 '14.9 '13.9 일반지주회사 162 157 154 157 163 164 183 152 130 117 114 금융지주회사 10 10 10 10 10 9 10 10 10 15 13 합 계 172 167 164 167 173 173 193 162 140 132 127 증가율 3.0 1.8 -1.8 -3.5 - -10.4 19.4 15.7 6.1 3.9 10.4 출처 : '23년 공정거래법상 지주회사 현황 분석결과 (공정거래위원회) 가.
영업개황 당사는 2021년 5월 1일을 분할기일로 패션사업부문인 신설회사 (주)에프앤에프를 설립하고, 존속회사는 (주)에프앤에프홀딩스로 상호를 변경하며 인적 분할하였습니다.
또한 2021년 8월 19일 현물출자 유상증자를 통해 지주회사 요건을 충족하고, 2